# 04. 스토리지 백엔드

Deep Agents의 **스토리지 백엔드**는 에이전트가 파일을 읽고 쓰는 저장소 계층을 추상화합니다.
백엔드를 바꾸는 것만으로 에이전트의 파일 시스템을 메모리, 로컬 디스크, 영속 스토어 등으로 전환할 수 있습니다.

## 학습 목표
- 백엔드가 에이전트의 파일 시스템을 어떻게 구현하는지 이해한다
- 5가지 내장 백엔드의 특성과 사용 시나리오를 파악한다
- `CompositeBackend`로 경로별 백엔드 라우팅을 구성한다
- `BackendProtocol`을 구현하여 커스텀 백엔드를 만든다

In [ ]:
# 환경 설정: .env 파일에서 API 키를 로드하고 LLM 모델을 초기화합니다.
from dotenv import load_dotenv
import os

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY가 설정되지 않았습니다!"
print("환경 설정 완료")

from langchain_openai import ChatOpenAI

# gpt-5.4: Deep Agents 에이전트의 추론에 사용할 LLM
model = ChatOpenAI(model="gpt-5.4")

In [ ]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")
# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


---
## 1. 백엔드란?

Deep Agents의 빌트인 파일 도구(`ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`)는  
모두 **백엔드(Backend)** 를 통해 동작합니다.

백엔드는 에이전트가 파일을 읽고 쓰는 **스토리지 계층**을 추상화합니다.  
같은 도구 호출이라도 백엔드에 따라 메모리에 저장될 수도, 로컬 디스크에 기록될 수도,  
원격 스토어에 영속화될 수도 있습니다.

![백엔드 추상화](../assets/images/backend_abstraction.png)

### 사용 가능한 백엔드

| 백엔드 | 저장 위치 | 영속성 | 사용 시나리오 |
|--------|----------|--------|-------------|
| `StateBackend` | 에이전트 상태 (메모리) | 스레드 내 | 임시 작업, 스크래치패드 (기본값) |
| `FilesystemBackend` | 로컬 디스크 | 영구 | 로컬 파일 접근, 코딩 에이전트 |
| `StoreBackend` | LangGraph Store | 크로스 스레드 | 장기 메모리, 사용자 선호도 |
| `CompositeBackend` | 경로별 라우팅 | 혼합 | 메모리 + 임시 파일 병용 |
| `LocalShellBackend` | 디스크 + 셸 | 영구 | 개발 환경 (보안 주의) |

In [ ]:
# 5가지 내장 백엔드 클래스와 프로토콜 인터페이스를 임포트합니다.
# BackendProtocol: 커스텀 백엔드를 만들 때 구현해야 하는 인터페이스
from deepagents.backends import (
    StateBackend,
    FilesystemBackend,
    StoreBackend,
    CompositeBackend,
)
from deepagents.backends.protocol import BackendProtocol

print("모든 백엔드 클래스를 성공적으로 임포트했습니다!")

---
## 2. StateBackend (기본값)

에이전트 상태(LangGraph state)에 파일을 저장합니다.  
**에페메럴(ephemeral)**: 대화 스레드 내에서만 파일이 유지됩니다.

### 특징
- `create_deep_agent()`에서 `backend`를 지정하지 않으면 자동 사용
- 파일이 체크포인트를 통해 에이전트 턴 간에는 유지됨
- 스레드가 종료되면 파일 소멸
- 외부 스토리지 불필요 — 추가 설정 없이 바로 사용

### 적합한 경우
- 일회성 파일 생성/분석 작업
- 대화 중 임시 스크래치패드가 필요한 경우
- 빠른 프로토타이핑

In [ ]:
from deepagents import create_deep_agent

# StateBackend는 기본값이므로 backend 파라미터 생략 시 자동 적용됩니다.
# 에이전트가 write_file, ls 등의 도구를 사용하면 메모리 내 가상 파일 시스템에서 동작합니다.
agent = create_deep_agent(
    model=model,
    system_prompt="당신은 파일 관리를 도와주는 어시스턴트입니다. 한국어로 응답하세요.",
)

# 에이전트에게 파일 작성 요청
# → 내부적으로 write_file, ls 도구가 StateBackend의 메모리에서 동작합니다.
result = agent.invoke(
    {"messages": [{"role": "user", "content": "'안녕하세요.txt' 파일을 만들고 '안녕하세요!'라고 적어 주세요. 그런 다음 파일 목록을 확인해 주세요."}]},
    config=lf_config,
)

print(result["messages"][-1].content)

---
## 3. FilesystemBackend — 로컬 디스크 접근

에이전트가 **실제 로컬 파일 시스템**에 접근할 수 있게 합니다.

### 주요 옵션
- `root_dir` — 접근 가능한 루트 디렉토리 (기본: 현재 디렉토리)
- `virtual_mode=True` — 경로 제한 활성화 (`..`, `~` 등으로 루트 밖 탈출 차단)
- `max_file_size_mb` — 읽을 수 있는 최대 파일 크기

### ⚠️ 보안 주의사항
> `FilesystemBackend`는 에이전트에게 실제 파일 시스템 접근 권한을 부여합니다.  
> 프로덕션 환경에서는 반드시 `virtual_mode=True`를 사용하거나, 컨테이너 등 샌드박스를 고려하세요.

In [ ]:
import tempfile
from pathlib import Path

# 안전한 테스트를 위해 임시 디렉토리를 생성합니다.
# 실제 환경에서는 프로젝트 디렉토리를 root_dir로 지정합니다.
with tempfile.TemporaryDirectory() as tmp_dir:
    # 테스트용 샘플 파일 생성
    Path(tmp_dir, "sample.txt").write_text("이것은 샘플 파일입니다.", encoding="utf-8")
    Path(tmp_dir, "data.csv").write_text("이름,나이\n홍길동,30\n김영희,25", encoding="utf-8")

    # FilesystemBackend를 사용하는 에이전트 생성
    # virtual_mode=True: 에이전트가 root_dir 바깥으로 접근하는 것을 차단합니다.
    fs_agent = create_deep_agent(
        model=model,
        system_prompt="당신은 파일 분석 어시스턴트입니다. 한국어로 응답하세요.",
        backend=FilesystemBackend(
            root_dir=tmp_dir,
            virtual_mode=True,  # 경로 제한 활성화
        ),
    )

    # 에이전트가 실제 디스크의 파일을 읽고 분석합니다.
    result = fs_agent.invoke(
        {"messages": [{"role": "user", "content": "현재 디렉토리의 파일 목록을 보여주고, 각 파일의 내용을 읽어서 요약해 주세요."}]},
        config=lf_config,
)

    print(result["messages"][-1].content)

---
## 4. StoreBackend — 크로스 스레드 영속 저장소

LangGraph의 `BaseStore`를 활용하여 **대화 스레드를 넘어서** 파일을 영속적으로 저장합니다.

### 특징
- 다른 스레드에서도 같은 파일에 접근 가능 (크로스 스레드)
- Redis, PostgreSQL 등 다양한 스토어 구현 지원
- LangSmith 배포 시 자동 프로비저닝
- `assistant_id` 기반 네임스페이스로 에이전트 간 격리

### StateBackend vs StoreBackend
| | StateBackend | StoreBackend |
|---|---|---|
| 스레드 종료 후 | 파일 소멸 | 파일 유지 |
| 다른 스레드에서 접근 | 불가 | 가능 |
| 추가 설정 | 불필요 | `store` + `checkpointer` 필요 |

In [ ]:
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import MemorySaver

# InMemoryStore: 개발/테스트용 인메모리 스토어
# 프로덕션에서는 PostgresStore, RedisStore 등을 사용합니다.
store = InMemoryStore()
# MemorySaver: 에이전트 체크포인트를 메모리에 저장 (상태 복원용)
checkpointer = MemorySaver()

# StoreBackend는 런타임 정보가 필요하므로 팩토리 함수(lambda)로 전달합니다.
# runtime 객체에는 store, thread_id 등의 정보가 포함됩니다.
store_agent = create_deep_agent(
    model=model,
    system_prompt="당신은 메모를 관리하는 어시스턴트입니다. 한국어로 응답하세요.",
    backend=lambda runtime: StoreBackend(runtime),
    store=store,
    checkpointer=checkpointer,
)

print("StoreBackend 에이전트가 생성되었습니다!")

In [ ]:
# 스레드 1에서 메모 저장 — thread_id로 대화 스레드를 구분합니다.
config_thread1 = {"configurable": {"thread_id": "thread-1"}}

result1 = store_agent.invoke(
    {"messages": [{"role": "user", "content": "'memo.txt' 파일을 만들고, '중요: 회의는 매주 월요일 오전 10시'라고 적어 주세요."}]},
    config={**config_thread1, **lf_config},
)
print("[스레드 1] 결과:")
print(result1["messages"][-1].content)
print()

In [ ]:
# 스레드 2에서 같은 메모 읽기
# StoreBackend 덕분에 다른 스레드에서도 동일한 파일에 접근할 수 있습니다.
# 이것이 StateBackend와의 핵심 차이점입니다.
config_thread2 = {"configurable": {"thread_id": "thread-2"}}

result2 = store_agent.invoke(
    {"messages": [{"role": "user", "content": "'memo.txt' 파일이 있나요? 있으면 내용을 읽어 주세요."}]},
    config={**config_thread2, **lf_config},
)
print("[스레드 2] 결과:")
print(result2["messages"][-1].content)

---
## 5. CompositeBackend — 경로별 라우팅

서로 다른 경로를 서로 다른 백엔드로 라우팅합니다.  
가장 일반적인 패턴: **`/memories/*`는 영속 저장**, **나머지는 에페메럴**

![CompositeBackend 경로별 라우팅](../assets/images/composite_backend.png)

### 동작 원리
1. 파일 경로가 `routes`의 프리픽스와 매칭되는지 확인
2. 매칭되면 해당 백엔드로 라우팅 (예: `/memories/prefs.txt` → StoreBackend)
3. 매칭되지 않으면 `default` 백엔드 사용 (예: `/scratch.txt` → StateBackend)

In [ ]:
# CompositeBackend 팩토리 함수
# 런타임 정보를 받아서 경로 기반 라우팅 백엔드를 생성합니다.
def create_composite_backend(runtime):
    """경로 기반 라우팅 백엔드 생성"""
    return CompositeBackend(
        default=StateBackend(runtime),           # 기본: 에페메럴 (스레드 종료 시 소멸)
        routes={
            "/memories/": StoreBackend(runtime),  # /memories/* → 영속 저장 (크로스 스레드)
        },
    )


composite_store = InMemoryStore()
composite_checkpointer = MemorySaver()

composite_agent = create_deep_agent(
    model=model,
    system_prompt="""당신은 메모 관리 어시스턴트입니다.
- 영구 저장이 필요한 메모는 /memories/ 경로에 저장하세요.
- 임시 작업 파일은 루트(/) 경로에 저장하세요.
한국어로 응답하세요.""",
    backend=create_composite_backend,
    store=composite_store,
    checkpointer=composite_checkpointer,
)

print("CompositeBackend 에이전트가 생성되었습니다!")

In [ ]:
# CompositeBackend 테스트 — 영속 vs 에페메럴 경로 비교
# /memories/preferences.txt → StoreBackend (영속 저장, 다른 스레드에서도 접근 가능)
# /scratch.txt → StateBackend (에페메럴, 이 스레드에서만 존재)
config = {"configurable": {"thread_id": "composite-test"}}

result = composite_agent.invoke(
    {"messages": [{"role": "user", "content": """
두 가지 파일을 만들어 주세요:
1. /memories/preferences.txt — '선호 언어: Python, 에디터: VS Code'
2. /scratch.txt — '임시 메모: 내일 할 일 정리'
그리고 두 파일이 모두 잘 생성되었는지 확인해 주세요.
"""}]},
    config={**config, **lf_config},
)

print(result["messages"][-1].content)

> **참고**: `CompositeBackend`는 라우트 프리픽스를 제거한 후 대상 백엔드에 저장합니다.  
> 예: `/memories/preferences.txt` → StoreBackend에는 `/preferences.txt`로 저장  
> 하지만 에이전트는 항상 전체 경로(`/memories/preferences.txt`)로 접근합니다.  
> 이 경로 변환은 CompositeBackend가 투명하게 처리합니다.

---
## 6. LocalShellBackend — 셸 실행

`LocalShellBackend`는 `FilesystemBackend`에 **셸 명령 실행 기능**(`execute` 도구)을 추가합니다.

### ⚠️ 보안 경고
> 호스트 시스템에서 **사용자 권한으로 명령이 직접 실행**됩니다.  
> 개발 환경에서만 사용하고, 프로덕션에서는 **샌드박스 백엔드**를 사용하세요.

```python
from deepagents.backends import LocalShellBackend

# ⚠️ 개발 환경에서만 사용하세요!
agent = create_deep_agent(
    model=model,
    backend=LocalShellBackend(root_dir="./workspace", virtual_mode=True),
    interrupt_on={"execute": True},  # 셸 명령은 승인 필요
)
```

> 이 노트북에서는 안전상의 이유로 `LocalShellBackend`를 직접 실행하지 않습니다.

---
## 7. 커스텀 백엔드 구현

`BackendProtocol`을 구현하면 나만의 백엔드를 만들 수 있습니다.

### 필수 메서드

| 메서드 | 설명 |
|--------|------|
| `ls_info(path)` | 디렉토리 내용 목록 |
| `read(file_path, offset, limit)` | 파일 읽기 (줄 번호 포함) |
| `write(file_path, content)` | 새 파일 생성 |
| `edit(file_path, old_string, new_string)` | 텍스트 교체 |
| `grep_raw(pattern, path, glob)` | 패턴 기반 파일 내용 검색 |
| `glob_info(pattern, path)` | 글로브 패턴으로 파일 검색 |

In [ ]:
# 간단한 커스텀 백엔드 예시: 읽기 전용 딕셔너리 기반
# BackendProtocol의 모든 필수 메서드를 구현합니다.
# 이 백엔드는 초기화 시 전달된 딕셔너리의 파일만 읽을 수 있고, 쓰기는 거부합니다.
from deepagents.backends.protocol import FileInfo, GrepMatch, WriteResult, EditResult


class ReadOnlyDictBackend:
    """딕셔너리에 파일을 저장하는 읽기 전용 백엔드 예시.
    
    BackendProtocol의 6가지 필수 메서드를 모두 구현합니다.
    쓰기/수정 요청 시 에러를 반환하여 읽기 전용을 강제합니다.
    """

    def __init__(self, files: dict[str, str]):
        self._files = files  # {파일경로: 파일내용} 딕셔너리

    def ls_info(self, path: str = "/") -> list[FileInfo]:
        """지정된 경로 아래의 파일 목록을 반환합니다."""
        return [
            {"path": p, "is_dir": False, "size": len(c), "modified_at": None}
            for p, c in self._files.items()
            if p.startswith(path)
        ]

    def read(self, file_path: str, offset: int = 0, limit: int = 2000) -> str:
        """파일 내용을 줄 번호와 함께 반환합니다 (cat -n 형식)."""
        content = self._files.get(file_path, "")
        lines = content.splitlines()
        selected = lines[offset:offset + limit]
        return "\n".join(f"{i + offset + 1}\t{line}" for i, line in enumerate(selected))

    def write(self, file_path: str, content: str) -> WriteResult:
        """쓰기 요청을 거부합니다 (읽기 전용)."""
        return WriteResult(error="읽기 전용 백엔드입니다.", path=None, files_update=None)

    def edit(self, file_path: str, old_string: str, new_string: str, replace_all: bool = False) -> EditResult:
        """수정 요청을 거부합니다 (읽기 전용)."""
        return EditResult(error="읽기 전용 백엔드입니다.", path=None, files_update=None, occurrences=None)

    def grep_raw(self, pattern: str, path: str | None = None, glob: str | None = None) -> list[GrepMatch]:
        """정규식 패턴으로 파일 내용을 검색합니다."""
        import re
        results = []
        for fpath, content in self._files.items():
            for i, line in enumerate(content.splitlines(), 1):
                if re.search(pattern, line):
                    results.append({"path": fpath, "line": i, "text": line})
        return results

    def glob_info(self, pattern: str, path: str = "/") -> list[FileInfo]:
        """글로브 패턴으로 파일을 검색합니다."""
        import fnmatch
        return [
            {"path": p, "is_dir": False, "size": len(c), "modified_at": None}
            for p, c in self._files.items()
            if fnmatch.fnmatch(p, pattern)
        ]


# 사용 예시: 두 개의 마크다운 문서를 가진 읽기 전용 백엔드
custom_backend = ReadOnlyDictBackend({
    "/docs/guide.md": "# 가이드\n이것은 가이드 문서입니다.\n## 설치 방법\npip install deepagents",
    "/docs/faq.md": "# FAQ\nQ: 지원하는 모델은?\nA: Anthropic, OpenAI 등 다양한 모델을 지원합니다.",
})

# 각 메서드 동작 확인
print("파일 목록:", custom_backend.ls_info("/"))
print()
print("파일 내용:")
print(custom_backend.read("/docs/guide.md"))
print()
print("검색 결과:", custom_backend.grep_raw("설치"))

---
## 백엔드 선택 가이드

아래 의사결정 트리를 참고하여 상황에 맞는 백엔드를 선택하세요.

![백엔드 선택 가이드](../assets/images/backend_decision_tree.png)

- **임시 작업만 필요** → `StateBackend` (기본값)
- **로컬 파일 읽기/쓰기** → `FilesystemBackend`
- **대화 간 데이터 유지** → `StoreBackend`
- **영속 + 임시 혼합** → `CompositeBackend`
- **셸 명령 실행까지 필요** → `LocalShellBackend` (개발 전용)

---
## 핵심 정리

| 백엔드 | 특징 | 파라미터 |
|--------|------|----------|
| `StateBackend` | 에페메럴, 기본값 | `backend` 생략 시 자동 |
| `FilesystemBackend` | 로컬 디스크 | `root_dir`, `virtual_mode` |
| `StoreBackend` | 크로스 스레드 영속 | `store` + `checkpointer` 필요 |
| `CompositeBackend` | 경로별 라우팅 | `default` + `routes` |
| `LocalShellBackend` | 디스크 + 셸 실행 | `root_dir` (보안 주의) |